# Does simulated disagreement predict real controversy?

A backtest for [Lightningfish](https://github.com/rajul-kk/LightningFish) that
scores something every previous run threw away.

## Why this exists

Every earlier backtest reduced a finished simulation to `sign(final mean opinion)`
— **one bit**, and the same bit a single LLM call produces. Scored that way, at
n=200, the result was unambiguous:

| | accuracy |
|---|---|
| author-karma heuristic | 62.5% |
| **simulation** | **51.5%** |
| single LLM call | 50.0% |
| majority class | 50.0% |

`p = 0.9994`. The simulation is at chance. It beats one raw model call by 1.5
points — a real but negligible contribution from all the agents and rounds.

Here is the problem with concluding "the engine is worthless" from that: we
evaluated a *population* on the one output where a population has no structural
advantage. A multi-agent run also produces a **distribution** — 24 opinions with
a spread — and nothing about the mean captures whether the crowd agreed.

This notebook tests the other moment. **Does the dispersion of simulated
opinion predict whether a real HN thread turned into an argument?**

## Honest framing before you run it

- This is **not** a claim a single call cannot make. You can just ask a model
  "will this be controversial", and the `single_llm` rung here does exactly
  that. The difference is that the simulation *derives* disagreement from
  population heterogeneity rather than asserting it. That is a weaker
  distinction than "structurally impossible", and I want it stated plainly.
- Controversy prediction is an established task (Reddit controversy scoring,
  Wikipedia edit wars). Doing it on HN is not novel in itself.
- Given the mean axis came out at chance, **expect this to fail too**. The
  point is to stop measuring the wrong thing, not to rescue a result.

---
## What the data is

**Source:** the [Hacker News Algolia API](https://hn.algolia.com/api) — free,
unauthenticated, ~10k requests/hour. No key, no scraping.

**Sample:** settled stories at least 24 hours old, pulled class-balanced (half
above the high points threshold, half below the low one) so the majority-class
baseline cannot be trivially high.

**The seed each agent reads** — strictly submission-time fields, never the
outcome:

| Field | Example |
|---|---|
| title | "China is now the world's greatest oil power" |
| author + karma | `bookofjoe` (110,566) |
| url domain | `economist.com` |
| type | story / Ask HN / Show HN |
| self-text | first 500 chars, if any |

**The label:** `num_comments / points` at settlement.

- ratio >= 0.7 -> **contested** (a thread arguing with itself)
- ratio < 0.4 -> **consensus** (quietly upvoted)
- in between -> skipped, no clean signal
- **fewer than 20 points -> skipped entirely**, because 0 comments on a 1-point
  story means nobody saw it, not that everyone agreed. This floor removes a lot
  of events, and is the main reason the usable sample is small.

Thresholds were placed either side of the observed median ratio to balance the
classes, *not* tuned against any model's accuracy.

Point-in-time safety is enforced in code, not by convention: the seed enricher
is a separate module from the ground-truth fetcher, and tests assert the target
values never appear in seed text or metadata.

---
## What the model is

**Inference:** [qwen2.5:7b](https://ollama.com/library/qwen2.5) (Q4_K_M, ~4.7 GB)
served locally by Ollama inside the notebook. No API keys, no external calls,
$0. It fits in a T4's 16 GB of VRAM, which is the entire reason this runs here
rather than locally — on a loaded 16 GB CPU box the same job thrashed the
pagefile and completed **zero** events in 80 minutes.

**The simulation** is not one prompt. Each event runs a population of 24 agents
over 3-4 rounds, and each round splits them into three tiers:

| Tier | Share | What happens |
|---|---|---|
| T1 originators | ~10% | LLM writes a structured post (stance, tag, confidence, blurb) |
| T2 reactors | ~20% | LLM re-evaluates after reading a feed of others' posts |
| T3 drifters | the rest | deterministic herding maths, no LLM call |

Only ~30% of agents call the model per round, which keeps a 24-agent, 4-round
run at roughly 26 calls.

**The agents** are six HN archetypes with different resistance, recency bias,
contrarian tendency and herding coefficients — from `CasualLurkerVoter`
(30%, low conviction, herds) to `GreybeardCynic` (12%, highest resistance,
anti-herds). Heterogeneity is the point: a population of identical agents
would converge trivially and its spread would carry no information.

**What is scored here:** the standard deviation of the 24 final opinions.
Above 0.35 -> predicted *contested*; below -> *consensus*. That threshold is a
fixed a-priori constant, not swept.

**The baseline ladder** — the simulation must beat every rung:

| Rung | What it controls for |
|---|---|
| majority class | degenerate data |
| `naive` | Ask-HN / question-mark heuristic, no model |
| `single_llm` | one call asked *the same controversy question* |
| the simulation | — |

---
## 1. Setup

Sidebar: **Accelerator -> GPU** and **Internet -> On**.

The `zstd` install is not optional — Ollama's current installer extracts with
it and Kaggle's image does not ship it, so without this the install fails
quietly and surfaces later as a confusing `FileNotFoundError: 'ollama'`.

In [ ]:
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd > /dev/null 2>&1
!zstd --version || echo "WARNING: zstd missing - the install below will fail"
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import shutil, subprocess, time, requests

if shutil.which("ollama") is None:
    raise RuntimeError(
        "ollama not found after install. Scroll up for the installer's error: "
        "usually zstd (cell above) or Internet disabled in the sidebar."
    )

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("ollama up"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama installed but the server did not start")

In [ ]:
MODEL = "qwen2.5:7b"

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ollama pull {MODEL}

# keep_alive=-1 pins the model in VRAM; otherwise it unloads after 5 idle
# minutes and reloads mid-run.
requests.post("http://localhost:11434/api/generate",
              json={"model": MODEL, "prompt": "hi", "stream": False, "keep_alive": -1},
              timeout=600)

for m in requests.get("http://localhost:11434/api/ps").json().get("models", []):
    vram = m.get("size_vram", 0) / 1e9
    print(f"{m['name']}: {vram:.2f} GB in VRAM")
    assert vram > 0, "model landed on CPU - enable the GPU accelerator, this is pointless otherwise"
print("GPU inference confirmed")

In [ ]:
!git clone --depth 1 https://github.com/rajul-kk/LightningFish.git /kaggle/working/lf
!pip -q install anthropic openai scipy requests pytest

import os, sys
os.chdir("/kaggle/working/lf")
sys.path.insert(0, "/kaggle/working/lf")

# Engine + HN suites only; finance/service tests pull deps an HN run never uses.
!python -m pytest tests/core tests/hn -q 2>&1 | tail -3

---
## 2. Configuration

`PULL_LIMIT` matters more here than in the other notebooks. The 20-point floor
plus the mid-ratio gap zone discard most stories, so a 40-story pull left only
**15 scoreable events** locally — far too few to conclude anything. Pull
generously.

In [ ]:
PULL_LIMIT = 300      # stories to pull; expect only ~1 in 3 to be scoreable
N_AGENTS   = 24       # population size - the spread of THIS is what's scored
N_ROUNDS   = 4

os.environ["LIGHTNINGFISH_MODEL"] = f"ollama:{MODEL}"
os.environ["LIGHTNINGFISH_N_AGENTS"] = str(N_AGENTS)
os.environ["LIGHTNINGFISH_N_ROUNDS"] = str(N_ROUNDS)
os.environ["LIGHTNINGFISH_LOCAL_TIMEOUT"] = "120"
os.environ["PYTHONUNBUFFERED"] = "1"
print(f"{MODEL} | {N_AGENTS} agents x {N_ROUNDS} rounds | pulling {PULL_LIMIT}")

### Throughput check

Roughly 26 model calls per event. Confirm the per-call cost before committing —
if this shows double digits you are on CPU regardless of what the assert said.

In [ ]:
import time
from lightningfish_core.llm_provider import make_provider

provider = make_provider(f"ollama:{MODEL}")
t0 = time.time()
for _ in range(3):
    provider.get_opinion("Output ONLY a number between -1 and 1.", "Rate: 0.5", f"ollama:{MODEL}")
per_call = (time.time() - t0) / 3
print(f"{per_call:.2f}s per call  ->  ~{per_call*26:.0f}s per event")

---
## 3. How many events are actually scoreable?

Runs the pull and the label filter **before** any simulation, so you find out
the usable sample size in seconds rather than after an hour of inference.

In [ ]:
from lightningfish_core.event_cache import CachingAdapter, EventCache, cached_pull_events
from lightningfish_hn.backtest_events import pull_hn_events
from lightningfish_hn.config import HNControversyAdapter

cache = EventCache("hn_stories")
adapter = CachingAdapter(HNControversyAdapter(), cache)
events = cached_pull_events(cache, f"hn:points:{PULL_LIMIT}",
                            lambda: pull_hn_events("points", PULL_LIMIT))

scoreable, contested, consensus = [], 0, 0
for ev in events:
    truth = adapter.get_ground_truth(ev.seed)
    if truth is None:
        continue
    d = adapter.truth_direction(truth)
    if d == 0:
        continue
    scoreable.append(ev)
    if d > 0:
        contested += 1
    else:
        consensus += 1

n = len(scoreable)
print(f"{n} scoreable of {len(events)} pulled  ({contested} contested / {consensus} consensus)")
print(f"majority-class floor: {max(contested, consensus)/n:.1%}" if n else "nothing scoreable")
print(f"estimated runtime: {per_call*26*n/60:.0f} min")
if n < 40:
    print("\nWARNING: under ~40 events the binomial test cannot detect anything "
          "short of a huge margin. Raise PULL_LIMIT before spending GPU time.")

---
## 4. Run it

Each finished simulation is cached (trajectory + final distribution), so
re-scoring later — against controversy, the mean, or anything else — costs
nothing. Not persisting these is exactly why this experiment needed a fresh
run instead of re-reading old ones.

In [ ]:
from lightningfish_core.engine import SimulationEngine
from lightningfish_core.models import SimulationResult

engine = SimulationEngine(adapter, model=f"ollama:{MODEL}")
run_key = f"ollama:{MODEL}:{N_AGENTS}x{N_ROUNDS}"

pairs, t0 = [], time.time()
for i, ev in enumerate(scoreable, 1):
    cached = cache.get_run(ev.event_id, run_key)
    if cached:
        result = SimulationResult(
            seed=ev.seed, trajectory=cached["trajectory"], round_events=[],
            final_distribution=cached["final_distribution"],
            total_tier1_calls=0, total_cost_usd=0.0,
            mean_parse_success_rate=cached["mean_parse_success_rate"],
            low_confidence=cached["low_confidence"],
        )
    else:
        result = engine.run(ev.seed, adapter.build_personas(N_AGENTS), n_rounds=N_ROUNDS)
        cache.put_run(ev.event_id, run_key, result)
        cache.save()
    pairs.append((ev, result))

    if i % 5 == 0 or i == len(scoreable):
        el = time.time() - t0
        print(f"{i}/{len(scoreable)}  {el/i:.0f}s/event  eta {el/i*(len(scoreable)-i)/60:.0f}m",
              flush=True)

print(f"done: {len(pairs)} events in {(time.time()-t0)/60:.1f} min")

---
## 5. Score against the ladder

In [ ]:
from lightningfish_core.backtest import llm_baseline, score_precomputed, sign

report = score_precomputed(adapter, pairs, baselines={
    "naive": lambda e: sign(adapter.naive_prediction(e.seed)),
    "single_llm": llm_baseline(adapter, engine),
})

print("=== hn controversy: does the crowd split? ===")
print(f"  n={report.n_events}  sim={report.sim_accuracy:.1%}  "
      f"majority={report.majority_class_accuracy:.1%}")
for name, acc in report.baseline_accuracy.items():
    print(f"  vs {name:<12} {acc:>6.1%}   {'PASS' if report.beats_baselines[name] else 'FAIL'}")
print(f"  p_value_vs_best = {report.p_value_vs_best:.4f}")
print(f"  parse_rate={report.mean_parse_success_rate:.2f}  skipped={report.skipped}")

### Is the predictor even varying?

A constant prediction can still post a decent-looking accuracy when the classes
are imbalanced. If the simulation says "contested" for everything, its spread
carries no information and the accuracy number is meaningless — worth checking
explicitly, because the mean-axis runs failed in exactly this way (the sim
predicted "viral" on 21 of 22 stories).

In [ ]:
import statistics

spreads = []
for ev, res in pairs:
    d = res.final_distribution
    if len(d) >= 2:
        m = sum(d) / len(d)
        spreads.append((sum((x - m) ** 2 for x in d) / len(d)) ** 0.5)

preds = [o.sim_direction for o in report.outcomes]
print(f"predicted contested: {preds.count(1)}/{len(preds)}   consensus: {preds.count(-1)}/{len(preds)}")
if spreads:
    print(f"stddev of final opinions: min={min(spreads):.3f} "
          f"median={statistics.median(spreads):.3f} max={max(spreads):.3f}")
    print("threshold in use: 0.35")
if len(set(preds)) == 1:
    print("\nThe simulation predicted ONE class for every event - its dispersion "
          "carries no signal here and the accuracy above is an artifact of the "
          "class balance, not a result.")

---
## 6. Save

In [ ]:
!cp -r .cache/lightningfish /kaggle/working/cache
!ls -la /kaggle/working/cache

The cache now holds each run's final distribution, so any future question about
these simulations can be answered without re-running them.

## Reading the result

1. **Parse rate first.** Below 0.8 and the run reflects malformed output.
2. **Check the predictor varies** (section 5b). A constant predictor invalidates
   the accuracy number regardless of what it says.
3. **Every rung must PASS**, and `p_value_vs_best` must be below 0.05.
4. **A negative is the expected outcome and still worth recording.** The mean
   axis is settled at chance across n=200; if dispersion also carries nothing,
   that closes the question of whether the standard evaluation was simply
   measuring the wrong output — and is a cleaner finding than leaving it open.

Either way the number belongs in
[ARCHITECTURE.md](https://github.com/rajul-kk/LightningFish/blob/main/ARCHITECTURE.md)
section 10, alongside the rest.